In [5]:
import random
import datetime
from collections import defaultdict

NUM_USERS = 1000
MALICIOUS_PERCENT = 0.03

# -------------------------------
# BASELINES
# -------------------------------
def generate_user_baselines():
    baselines = {}
    for user_id in range(NUM_USERS):
        baselines[user_id] = {
            "avg_transaction": random.randint(50, 500),
            "common_location": random.choice(["Texas", "New York", "California"]),
            "device": random.choice(["mobile", "desktop"])
        }
    return baselines

# -------------------------------
# LOG GENERATION
# -------------------------------
def generate_logs(baselines):
    logs = []
    malicious_users = set(random.sample(range(NUM_USERS), int(NUM_USERS * MALICIOUS_PERCENT)))

    for user_id in range(NUM_USERS):
        base = baselines[user_id]
        is_malicious = user_id in malicious_users

        log = {
            "user_id": user_id,
            "timestamp": datetime.datetime.now(),
            "login_attempts": random.randint(1, 3),
            "ip_type": random.choice(["normal", "vpn"]),
            "location": base["common_location"],
            "device": base["device"],
            "new_device": False,
            "transaction": int(random.gauss(base["avg_transaction"], 50)),
        }

        # Diverse attack patterns
        if is_malicious:
            attack_type = random.choice([
                "brute_force",
                "vpn_only",
                "transaction_spike",
                "impossible_travel",
                "stealth",
                "combo"
            ])

            if attack_type == "brute_force":
                log["login_attempts"] = random.randint(5, 10)

            elif attack_type == "vpn_only":
                log["ip_type"] = "vpn"

            elif attack_type == "transaction_spike":
                log["transaction"] = random.randint(5000, 15000)

            elif attack_type == "impossible_travel":
                log["location"] = "Germany"

            elif attack_type == "stealth":
                log["transaction"] = base["avg_transaction"] * 3

            elif attack_type == "combo":
                log["login_attempts"] = random.randint(5, 10)
                log["ip_type"] = "vpn"
                log["location"] = "Unknown"
                log["transaction"] = random.randint(8000, 20000)
                log["new_device"] = True

        logs.append(log)

    return logs

# -------------------------------
# DETECTION ENGINE
# -------------------------------
def analyze_behavior(log, baseline, user_events):
    risks = {}
    reasons = []

    # LOGIN
    login_risk = min(log["login_attempts"] / 6, 1)
    if login_risk > 0.7:
        reasons.append(f"{log['login_attempts']} failed login attempts")
    risks["login"] = login_risk

    # LOCATION
    location_risk = 1 if log["location"] != baseline["common_location"] else 0
    if location_risk:
        reasons.append(f"Login from {log['location']} instead of {baseline['common_location']}")
    risks["location"] = location_risk

    # VPN
    vpn_risk = 1 if log["ip_type"] == "vpn" else 0
    if vpn_risk:
        reasons.append("Access via VPN or suspicious IP")
    risks["vpn"] = vpn_risk

    # TRANSACTION
    avg = baseline["avg_transaction"]
    deviation = abs(log["transaction"] - avg) / (avg + 1)
    transaction_risk = min(deviation, 1)

    if transaction_risk > 0.8:
        reasons.append(f"Transaction spike: ${log['transaction']} vs avg ${avg}")
    risks["transaction"] = transaction_risk

    # DEVICE
    device_risk = 1 if log["new_device"] else 0
    if device_risk:
        reasons.append("Login from new/unrecognized device")
    risks["device"] = device_risk

    # TIME-BASED (VELOCITY)
    now = log["timestamp"]
    recent_events = user_events[log["user_id"]]

    rapid_events = [e for e in recent_events if (now - e).seconds < 60]
    velocity_risk = 1 if len(rapid_events) >= 3 else 0

    if velocity_risk:
        reasons.append(f"{len(rapid_events)} actions within 60 seconds")

    risks["velocity"] = velocity_risk

    # store timestamp
    user_events[log["user_id"]].append(now)

    return risks, reasons

# -------------------------------
# SCORING (IMPROVED)
# -------------------------------
def calculate_risk_score(risks):
    base_score = (
        0.25 * risks["login"] +
        0.20 * risks["transaction"] +
        0.15 * risks["location"] +
        0.15 * risks["vpn"] +
        0.10 * risks["device"] +
        0.15 * risks["velocity"]
    )

    # add slight randomness for realism
    noise = random.uniform(-0.05, 0.05)

    final_score = min(max(base_score + noise, 0), 1)

    return round(final_score * 100, 2)

# -------------------------------
# ALERT SYSTEM
# -------------------------------
def generate_alert(log, score, reasons):
    if score >= 80:
        severity = "CRITICAL"
    elif score >= 50:
        severity = "HIGH"
    else:
        return None

    confidence = round(score / 100, 2)

    return {
        "alert_id": random.randint(1000, 9999),
        "user_id": log["user_id"],
        "severity": severity,
        "score": score,
        "confidence": confidence,
        "reasons": reasons,
        "timestamp": log["timestamp"].strftime("%Y-%m-%d %H:%M:%S")
    }

# -------------------------------
# MAIN
# -------------------------------
def run_system():
    baselines = generate_user_baselines()
    logs = generate_logs(baselines)

    user_events = defaultdict(list)
    alerts = []

    print("Starting Cyber Threat Detection Simulation...\n")

    for log in logs:
        baseline = baselines[log["user_id"]]

        risks, reasons = analyze_behavior(log, baseline, user_events)
        score = calculate_risk_score(risks)

        alert = generate_alert(log, score, reasons)

        if alert:
            alerts.append(alert)

            print(f"--- [SOC ALERT] ---")
            print(f"Alert ID: {alert['alert_id']}")
            print(f"User: {alert['user_id']}")
            print(f"Severity: {alert['severity']} (Score: {alert['score']}, Confidence: {alert['confidence']})")
            print(f"Issues: {', '.join(alert['reasons'])}")
            print(f"Time: {alert['timestamp']}")
            print(f"Action: Investigate / MFA / Possible Lock\n")

    print(f"\nTotal Alerts Generated: {len(alerts)}")

# -------------------------------
# RUN
# -------------------------------
if __name__ == "__main__":
    run_system()

Starting Cyber Threat Detection Simulation...

--- [SOC ALERT] ---
Alert ID: 7529
User: 0
Severity: CRITICAL (Score: 88.78, Confidence: 0.89)
Issues: 7 failed login attempts, Login from Unknown instead of California, Access via VPN or suspicious IP, Transaction spike: $15986 vs avg $322, Login from new/unrecognized device
Time: 2026-05-17 15:44:47
Action: Investigate / MFA / Possible Lock

--- [SOC ALERT] ---
Alert ID: 5644
User: 166
Severity: CRITICAL (Score: 83.5, Confidence: 0.83)
Issues: 10 failed login attempts, Login from Unknown instead of Texas, Access via VPN or suspicious IP, Transaction spike: $19060 vs avg $319, Login from new/unrecognized device
Time: 2026-05-17 15:44:47
Action: Investigate / MFA / Possible Lock

--- [SOC ALERT] ---
Alert ID: 5378
User: 329
Severity: HIGH (Score: 52.01, Confidence: 0.52)
Issues: Login from Germany instead of New York, Access via VPN or suspicious IP
Time: 2026-05-17 15:44:47
Action: Investigate / MFA / Possible Lock

--- [SOC ALERT] ---
Al